In [5]:
import psycopg2
import pandas as pd

# Shared connection parameters — change these to match your setup
DB_CONFIG = {
    "host":     "localhost",
    "database": "bank_review",
    "user":     "admin",
    "password": "admin123"
}

In [6]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- restaurants table ---
# Create restaurants table
# SERIAL auto-increments the ID on every insert — no need to supply it manually
# UNIQUE ensures no two restaurants share the same name
cur.execute("""
    CREATE TABLE IF NOT EXISTS Banks  (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE,
         app_name VARCHAR(255)
    );
""")

# --- reviews table (must come after restaurants because of the foreign key) ---
# Create reviews table
# restaurant_id is a FOREIGN KEY — it must match an existing restaurants.restaurant_id
# This link is what allows us to JOIN the two tables later

cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id INT REFERENCES Banks(bank_id),
        review_text   TEXT,
        rating        INT,
        review_date   DATE,
        sentiment_label VARCHAR(50),
        identified_theme VARCHAR(255),
        source        VARCHAR(50)
    );
""")

conn.commit()
cur.close()
conn.close()

print("Tables created successfully.")

Tables created successfully.


In [7]:
import pandas as pd
import os

# Paths to the CSV files
clean_reviews_path = "data/processed/cleaned_bank_reviews.csv"
proc_reviews_path = "processed_reviews.csv"

# Load the data
df_clean = pd.read_csv(clean_reviews_path)

if os.path.exists(proc_reviews_path):
    df_proc = pd.read_csv(proc_reviews_path)
    
    # Greedy matching on review text to merge sentiment and themes
    clean_groups = {text: rows for text, rows in df_clean.groupby('review')}
    used_indices = set()
    merged_rows = []
    
    for _, proc_row in df_proc.iterrows():
        text = proc_row['review_text']
        if text in clean_groups:
            for _, clean_row in clean_groups[text].iterrows():
                clean_idx = clean_row.name
                if clean_idx not in used_indices:
                    used_indices.add(clean_idx)
                    # Merge row dictionaries
                    merged_row = clean_row.to_dict()
                    merged_row.update(proc_row.to_dict())
                    merged_rows.append(merged_row)
                    break
                    
    # Append any remaining clean reviews that weren't in df_proc (using defaults)
    for idx, clean_row in df_clean.iterrows():
        if idx not in used_indices:
            merged_row = clean_row.to_dict()
            merged_row['review_id'] = idx + 1
            merged_row['review_text'] = clean_row['review']
            merged_row['sentiment_label'] = 'Neutral'
            merged_row['identified_theme'] = 'General'
            merged_rows.append(merged_row)
            
    df_reviews = pd.DataFrame(merged_rows)
else:
    print("Warning: processed_reviews.csv not found. Using cleaned reviews with default sentiment.")
    df_reviews = df_clean.copy()
    df_reviews['review_id'] = df_reviews.index + 1
    df_reviews['review_text'] = df_reviews['review']
    df_reviews['sentiment_label'] = 'Neutral'
    df_reviews['identified_theme'] = 'General'

# Re-assign review_id to be a clean, unique sequence starting from 1
df_reviews['review_id'] = range(1, len(df_reviews) + 1)

# Ensure required columns are present
df_reviews['review_date'] = df_reviews.get('date', '2026-05-19')
df_reviews['source'] = df_reviews.get('source', 'Google Play')

# Get unique banks for the banks table
df_bank = df_reviews[['bank_id', 'bank_name']].drop_duplicates().dropna()

print(f"Reviews: {len(df_reviews)} rows")
print(f"Banks:   {len(df_bank)} rows")

Reviews: 1458 rows
Banks:   3 rows


In [8]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

print(df_reviews.columns)

# --- Insert banks ---
for _, row in df_bank.iterrows():
    cur.execute(
        """
        INSERT INTO banks (bank_id, bank_name)
        VALUES (%s, %s)
        ON CONFLICT (bank_id) DO UPDATE SET bank_name = EXCLUDED.bank_name;
        """,
        (int(row["bank_id"]), row["bank_name"])
    )

# --- Insert reviews ---
inserted_count = 0
for _, row in df_reviews.iterrows():
    cur.execute(
        """
        INSERT INTO reviews (review_id, bank_id, review_text, sentiment_label, identified_theme, rating, review_date, source)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (review_id) DO NOTHING;
        """,
        (
            int(row["review_id"]),
            int(row["bank_id"]),
            row["review_text"],
            row["sentiment_label"],
            row["identified_theme"],
            int(row["rating"]),
            row["review_date"],
            row["source"]
        )
    )
    inserted_count += 1

conn.commit()
cur.close()
conn.close()

print(f"Successfully inserted {inserted_count} rows into the reviews table.")

Index(['bank_id', 'bank_name', 'app', 'review', 'rating', 'date', 'thumbs',
       'clean_text', 'review_id', 'review_text', 'sentiment_label',
       'sentiment_score', 'identified_theme', 'review_date', 'source'],
      dtype='object')
Successfully inserted 1458 rows into the reviews table.
